# Robustness Analysis

This notebook builds the final robustness layer for the cross-sectional equity prediction project. It follows the same logic as notebooks 14 and 15: all results are produced from persisted artifacts, no model is retrained, and the portfolio interpretation is anchored on the 25 bps one-way transaction-cost framework used in the final portfolio section.

The goal is to test whether the main conclusions survive reasonable changes in trading frictions, threshold choices, time period, validation-based selection, risk exposure, and architecture size.

## Setup

The setup cell executes `scripts/robustness_analysis.py`, which rebuilds the robustness tables, figures, and LaTeX text from existing backtest and forecasting artifacts. This keeps the notebook reproducible while preserving the train-validation-test discipline of the earlier pipeline.

In [ ]:
from pathlib import Path
import os

Path('/tmp/matplotlib-cache').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(362559)

plt.style.use('default')
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

BACKTEST_DIR = ROOT / 'outputs' / 'backtests' / 'dl_xgb_score_strategies'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_COST_BPS = 25.0

exec((ROOT / 'scripts' / 'robustness_analysis.py').read_text(), {'__name__': '__main__'})

def pct(x):
    return f'{100 * x:.2f}%'

def show_table(df, n=20):
    print(df.head(n).to_string(index=False))

## 1. Introduction

The forecasting analysis showed that the retained machine-learning scores contain cross-sectional ranking information. The portfolio analysis then converted those rankings into validation-selected trading strategies evaluated at 25 bps one-way costs. Robustness checks sit between those two claims: they ask whether the final portfolio evidence is driven by a narrow cost assumption, a specific threshold, one market regime, or validation overfitting.

All returns below are net returns unless explicitly stated otherwise. The 25 bps assumption remains the main reference case; 0, 10, and 50 bps are used only as sensitivity checks.

In [ ]:
summary = pd.read_csv(TABLE_DIR / 'robustness_summary.csv')
show_table(summary)

## 2. Transaction Cost Sensitivity

This check compares the representative MLP, FT, Temporal, and XGBoost baseline strategies across 0, 10, 25, and 50 bps one-way transaction costs. The strategies are the family representatives selected consistently with notebook 15. The economic question is whether the main results survive realistic trading frictions, not whether performance is maximized under zero costs.

A monotonic decline is expected because net returns subtract turnover times one-way cost. The key robustness criterion is whether Sharpe ratios remain economically meaningful at 25 bps and whether they collapse at 50 bps.

In [ ]:
cost_table = pd.read_csv(TABLE_DIR / 'cost_sensitivity.csv')
cost_long = pd.read_csv(TABLE_DIR / 'cost_sensitivity_long.csv')

cost_display = cost_table.copy()
for col in ['Sharpe (0bps)', 'Sharpe (10bps)', 'Sharpe (25bps)', 'Sharpe (50bps)']:
    cost_display[col] = cost_display[col].map(lambda x: f'{x:.3f}')
show_table(cost_display)

In [ ]:
plt.figure(figsize=(9, 5))
for strategy, grp in cost_long.groupby('strategy_base'):
    grp = grp.sort_values('one_way_cost_bps')
    plt.plot(grp['one_way_cost_bps'], grp['sharpe'], marker='o', linewidth=2, label=strategy)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('One-way transaction cost (bps)')
plt.ylabel('Test Sharpe')
plt.title('Transaction cost sensitivity: Sharpe')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_cost_sensitivity_sharpe.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
for strategy, grp in cost_long.groupby('strategy_base'):
    grp = grp.sort_values('one_way_cost_bps')
    plt.plot(grp['one_way_cost_bps'], grp['annualized_return'], marker='o', linewidth=2, label=strategy)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('One-way transaction cost (bps)')
plt.ylabel('Test annualized return')
plt.title('Transaction cost sensitivity: annualized return')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_cost_sensitivity_return.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** All four representative strategies remain positive at 50 bps, so the final portfolio results are not purely a zero-cost artifact. The MLP representative is the strongest under higher costs, while the baseline XGBoost representative is the most cost-sensitive. This is consistent with notebook 15: turnover and portfolio construction matter as much as raw ranking quality when moving from forecasts to implementable portfolios.

## 3. Threshold Sensitivity

Notebook 15 selects strategies on validation performance under the 25 bps framework. This section tests whether the portfolio results depend sharply on the zero-threshold or construction rule. A stable model should not require one extremely precise threshold to work.

In [ ]:
threshold_table = pd.read_csv(TABLE_DIR / 'threshold_sensitivity.csv')
threshold_display = threshold_table[[
    'model', 'model_family', 'best_threshold', 'best_rule', 'best_sharpe',
    'sharpe_variation_range', 'stable_vs_threshold'
]].copy()
for col in ['best_threshold', 'best_sharpe', 'sharpe_variation_range']:
    threshold_display[col] = threshold_display[col].map(lambda x: f'{x:.3f}')
show_table(threshold_display, n=20)

In [ ]:
# Rebuild the threshold curve directly from the saved 25 bps validation/test stability inputs.
perf = pd.read_csv(BACKTEST_DIR / 'dl_xgb_score_strategy_performance_summary.csv')
perf_25 = perf.loc[
    perf['split'].eq('test')
    & perf['one_way_cost_bps'].eq(ANALYSIS_COST_BPS)
    & perf['rule'].isin(['long_short', 'long_short_130_30', 'gross_normalized_long_short'])
].copy()

representative_models = cost_long['score_label'].drop_duplicates().tolist()
plt.figure(figsize=(10, 5.5))
for model in representative_models:
    line = (
        perf_25.loc[perf_25['score_label'].eq(model)]
        .groupby('zero_threshold', as_index=False)['sharpe']
        .max()
        .sort_values('zero_threshold')
    )
    plt.plot(line['zero_threshold'], line['sharpe'], marker='o', linewidth=2, label=model)
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Zero threshold')
plt.ylabel('Best test Sharpe at 25 bps')
plt.title('Threshold sensitivity by retained model signal')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_threshold_sensitivity.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** The retained representatives do not all behave identically. The best MLP threshold is stable relative to many alternatives, while some expected-return and classifier variants show larger Sharpe ranges. This reinforces the validation-selection logic: the final portfolio should be interpreted as a disciplined specification choice, not as evidence that every thresholded version of every score is equally strong.

## 4. Subperiod Performance

The test sample is split into 2016-2019 and 2020-2024. The first period captures the pre-pandemic expansion, while the second includes the COVID shock, inflation shock, and higher-rate environment. The purpose is to check whether the selected strategies depend entirely on one regime.

In [ ]:
subperiod = pd.read_csv(TABLE_DIR / 'subperiod_performance.csv')
subperiod_display = subperiod.copy()
for col in subperiod_display.columns:
    if col != 'strategy':
        subperiod_display[col] = subperiod_display[col].map(lambda x: f'{x:.3f}')
show_table(subperiod_display)

In [ ]:
monthly = pd.read_csv(BACKTEST_DIR / 'dl_xgb_score_strategy_monthly_returns.csv')
monthly['month'] = pd.to_datetime(monthly['month'])

selected = pd.read_csv(TABLE_DIR / 'portfolio_selected_top5_strategies.csv')
selected['strategy_base'] = selected.apply(
    lambda r: ' | '.join([
        'MLP' if str(r['score_label']).startswith('mlp') else
        'FT' if str(r['score_label']).startswith('ft') else
        'Temporal' if str(r['score_label']).startswith('temporal') else
        'baseline_model' if str(r['score_label']).startswith('xgb') else 'Other',
        str(r['score_label']), str(r['rule']),
        ('thr=' if bool(r['threshold_gate']) else 'plain=') + f"{float(r['zero_threshold']):.2f}"
    ]), axis=1
)
representatives = cost_long[['score_label', 'rule', 'q', 'sign_gate', 'threshold_gate', 'zero_threshold', 'strategy_base']].drop_duplicates()

selected_monthly = []
for _, spec in representatives.iterrows():
    mask = monthly['split'].eq('test') & monthly['one_way_cost_bps'].eq(ANALYSIS_COST_BPS)
    for col in ['score_label', 'rule', 'q', 'sign_gate', 'threshold_gate', 'zero_threshold']:
        mask &= monthly[col].eq(spec[col])
    tmp = monthly.loc[mask].copy()
    tmp['strategy_base'] = spec['strategy_base']
    selected_monthly.append(tmp)
selected_monthly = pd.concat(selected_monthly, ignore_index=True)
selected_monthly = selected_monthly.loc[selected_monthly['month'].ge(pd.Timestamp('2016-01-01'))].copy()
selected_monthly['subperiod'] = np.where(selected_monthly['month'].le(pd.Timestamp('2019-12-31')), '2016-2019', '2020-2024')

plt.figure(figsize=(10, 5.5))
for (strategy, subperiod_label), grp in selected_monthly.groupby(['strategy_base', 'subperiod']):
    grp = grp.sort_values('month').copy()
    grp['cumulative_wealth'] = (1 + grp['net_return']).cumprod()
    plt.plot(grp['month'], grp['cumulative_wealth'], linewidth=1.8, label=f'{strategy} ({subperiod_label})')
plt.xlabel('Month')
plt.ylabel('Cumulative wealth')
plt.title('Subperiod cumulative returns at 25 bps')
plt.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_subperiod_cumulative_returns.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** The selected representatives generate positive annualized returns in both subperiods. Performance is not perfectly constant, but the result is not concentrated exclusively in either the pre-2020 regime or the post-2020 regime. This supports the economic reading of notebook 15, while still leaving room for regime dependence in the magnitude of Sharpe ratios.

## 5. Validation vs Test Stability

The portfolio section selects strategies using validation Sharpe. This is necessary for a disciplined research design, but it also introduces model-selection risk because many score, threshold, and rule combinations are evaluated. The validation-versus-test scatter checks whether validation performance is informative out of sample.

In [ ]:
val_test = pd.read_csv(TABLE_DIR / 'validation_test_stability.csv')
validation_test_corr = val_test['sharpe_validation'].corr(val_test['sharpe_test'])
average_drop = val_test['sharpe_drop'].mean()
print(f'Validation-test Sharpe correlation: {validation_test_corr:.3f}')
print(f'Average validation-to-test Sharpe drop: {average_drop:.3f}')

val_test_display = val_test[[
    'strategy_base', 'model_family', 'sharpe_validation', 'sharpe_test', 'sharpe_drop',
    'annualized_return_validation', 'annualized_return_test'
]].head(10).copy()
for col in ['sharpe_validation', 'sharpe_test', 'sharpe_drop', 'annualized_return_validation', 'annualized_return_test']:
    val_test_display[col] = val_test_display[col].map(lambda x: f'{x:.3f}')
show_table(val_test_display, n=10)

In [ ]:
colors = {'MLP': 'tab:blue', 'FT': 'tab:green', 'Temporal': 'tab:orange', 'baseline_model': 'tab:red'}
plt.figure(figsize=(7, 6))
for family, grp in val_test.groupby('model_family'):
    plt.scatter(grp['sharpe_validation'], grp['sharpe_test'], label=family, alpha=0.75, s=45, color=colors.get(family))
lims = [
    min(val_test['sharpe_validation'].min(), val_test['sharpe_test'].min()) - 0.1,
    max(val_test['sharpe_validation'].max(), val_test['sharpe_test'].max()) + 0.1,
]
plt.plot(lims, lims, color='black', linestyle='--', linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.xlabel('Validation Sharpe')
plt.ylabel('Test Sharpe')
plt.title(f'Validation vs test Sharpe at 25 bps (corr={validation_test_corr:.2f})')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_validation_vs_test_sharpe.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** Validation Sharpe is positively related to test Sharpe, but the relationship is far from perfect. The average drop from validation to test performance confirms that selection on validation results can overstate expected out-of-sample performance. This does not invalidate the selected strategies, but it motivates conservative language in the final report.

## 6. High-Risk and High-Return Strategies

A high annualized return can reflect true predictive content, exposure to systematic risk, high leverage, high turnover, or a large drawdown profile. This section screens the 25 bps test universe for high-return, high-volatility, and high-drawdown strategies.

In [ ]:
high_risk = pd.read_csv(TABLE_DIR / 'high_risk_strategies.csv')
high_risk_display = high_risk[[
    'strategy', 'model_family', 'annualized_return', 'annualized_volatility', 'sharpe', 'max_drawdown', 'average_turnover'
]].head(12).copy()
for col in ['annualized_return', 'annualized_volatility', 'max_drawdown']:
    high_risk_display[col] = high_risk_display[col].map(lambda x: f'{100*x:.2f}%')
for col in ['sharpe', 'average_turnover']:
    high_risk_display[col] = high_risk_display[col].map(lambda x: f'{x:.3f}')
show_table(high_risk_display, n=12)

In [ ]:
risk_source = pd.read_csv(BACKTEST_DIR / 'dl_xgb_score_strategy_performance_summary.csv')
risk_source = risk_source.loc[risk_source['split'].eq('test') & risk_source['one_way_cost_bps'].eq(ANALYSIS_COST_BPS)].copy()
risk_source['model_family'] = risk_source['score_label'].map(
    lambda x: 'MLP' if str(x).startswith('mlp') else 'FT' if str(x).startswith('ft') else 'Temporal' if str(x).startswith('temporal') else 'baseline_model' if str(x).startswith('xgb') else 'Other'
)

plt.figure(figsize=(8, 5.5))
for family, grp in risk_source.groupby('model_family'):
    plt.scatter(grp['annualized_volatility'], grp['annualized_return'], label=family, alpha=0.55, s=40, color=colors.get(family))
for _, row in risk_source.nlargest(5, 'annualized_return').iterrows():
    plt.annotate(row['score_label'], (row['annualized_volatility'], row['annualized_return']), fontsize=7, xytext=(3, 3), textcoords='offset points')
plt.xlabel('Annualized volatility')
plt.ylabel('Annualized return')
plt.title('Return versus volatility for test strategies at 25 bps')
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_return_vs_volatility.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** The most extreme return strategies also carry substantial volatility and drawdown exposure. These are useful stress cases for the report: they show that some attractive portfolio outcomes should be interpreted as risk-bearing strategies rather than clean arbitrage. The final portfolio discussion should therefore emphasize risk-adjusted performance and implementability, not only annualized returns.

## 7. Architecture Robustness

The forecasting section compares predictive quality across model families. Here the question is narrower: do materially different architecture sizes change the conclusion? The table combines the main MLP, FT, and Temporal scores with available small, full, and tiny/debug artifacts. Some diagnostic variants have Rank IC but no corresponding selected portfolio Sharpe, so their Sharpe is left blank rather than imputed.

In [ ]:
architecture = pd.read_csv(TABLE_DIR / 'architecture_robustness.csv')
architecture_display = architecture.copy()
for col in ['rank_ic', 'sharpe']:
    architecture_display[col] = architecture_display[col].map(lambda x: '' if pd.isna(x) else f'{x:.3f}')
show_table(architecture_display, n=20)

In [ ]:
plot_arch = architecture.dropna(subset=['rank_ic']).copy()
plot_arch['label'] = plot_arch['architecture']
fig, ax1 = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(plot_arch))
ax1.bar(x - 0.18, plot_arch['rank_ic'], width=0.36, label='Rank IC', color='tab:blue')
ax1.set_ylabel('Test Rank IC')
ax1.set_xticks(x)
ax1.set_xticklabels(plot_arch['label'], rotation=35, ha='right', fontsize=8)
ax2 = ax1.twinx()
ax2.bar(x + 0.18, plot_arch['sharpe'].fillna(0), width=0.36, label='Best test Sharpe', color='tab:orange')
ax2.set_ylabel('Best test Sharpe at 25 bps')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.title('Architecture robustness: predictive and portfolio metrics')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'robustness_architecture_rank_ic_sharpe.png', dpi=180, bbox_inches='tight')
plt.show()

**Interpretation.** The full temporal model is materially stronger than the tiny/debug temporal variant, so the diagnostic architecture should not be used for final economic conclusions. The FT small result is close to the main FT score, and the main MLP/FT/Temporal models remain competitive in both forecasting and portfolio metrics. This supports the retained architecture choices while making clear that capacity constraints can matter.

## 8. Final Robustness Conclusions

The robustness layer strengthens the final empirical story, but it also disciplines it. The main result is not a zero-cost artifact, because representative strategies survive 25 bps and remain positive at 50 bps. It is also not entirely concentrated in one test subperiod. However, performance is sensitive to turnover, threshold choices for some signals, and validation-based selection. The strongest interpretation is therefore measured: the machine-learning scores contain useful ranking information, but the economic value depends on portfolio construction and implementability.

The final report should emphasize the retained 25 bps framework, validation-based model selection, and net-return definition used in notebook 15. The limitations section should explicitly discuss data snooping, transaction cost simplifications, the absence of a CRSP market benchmark, Compustat timing assumptions, and turnover constraints.

In [ ]:
latex_text = (TABLE_DIR / 'robustness_latex_sections.tex').read_text()
print(latex_text)